# LinkedIn Campaign Click-Through-Rate Analysis

## 1. Setup

In [32]:
# 1.1 Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1.2 Statistical tests
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.multitest import multipletests

# 1.3 Inline plotting in Jupyter Lab
%matplotlib inline

## 2. Data Loading & Cleaning

In [ ]:
# 2.1 Discover your LinkedIn reports anywhere under the notebook’s working dir
from pathlib import Path
import os

data_dir = Path(os.environ.get('LINKEDIN_REPORT_DIR', 'data')).expanduser()
paths = sorted(data_dir.glob('*creative_performance_report*.csv'))
if not paths:
    raise FileNotFoundError(f"No creative performance reports found in {data_dir}. See README.md for setup.")

# 2.2 Loader for UTF-16-LE TSV exports
def load_report(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', skiprows=5, encoding='utf-16-le')

# 2.3 Load & concatenate
df = pd.concat((load_report(p) for p in paths), ignore_index=True)

# 2.4 Normalize column names
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r'[\s()+%-]+', '_', regex=True)
      .str.replace(r'_{2,}', '_', regex=True)
      .str.strip('_')
)

# 2.5 Parse the required date column
if 'start_date_in_utc' not in df.columns:
    raise ValueError('Reports must include Start Date (in UTC).')
if 'start_date_in_utc' in df.columns:
    df['date'] = pd.to_datetime(df['start_date_in_utc'], format='%m/%d/%Y')

# 2.6 Convert “XX%” → float and comma-numbers → float
for c in df.columns:
    if df[c].dtype == object:
        s = df[c].astype(str)
        if s.str.contains('%', na=False).any():
            df[c] = s.str.rstrip('%').astype(float) / 100
        elif s.str.match(r'^[\d,]+(\.\d+)?$', na=False).all():
            df[c] = s.str.replace(',', '').astype(float)

# 2.7 Bring key identifiers to front
keys = ['date','account_name','campaign_group_name','campaign_name','creative_name']
df = df[[k for k in keys if k in df.columns] + [c for c in df.columns if c not in keys]]

# 2.8 Preview
df.head()

## 3. Exploratory CTR Summary

In [ ]:
from analysis import campaign_summary, pairwise_comparisons

agg = campaign_summary(df)
if agg.empty:
    raise ValueError('No campaigns have positive impressions.')
agg.reset_index()


## 4. Visualize Campaign CTRs

In [ ]:
plt.figure(figsize=(8,5))
plt.barh(
    agg.index,
    agg['ctr'],
    xerr=[(agg['ctr'] - agg['ci_lower']).clip(lower=0), (agg['ci_upper'] - agg['ctr']).clip(lower=0)],
    align='center'
)
plt.yticks(ticks=range(len(agg)), labels=agg.index)
plt.xlabel('Click-Through Rate')
plt.title('Campaign CTRs with 95% CI')
plt.gca().invert_yaxis()  # highest CTR on top
plt.tight_layout()
plt.show()

## 5. Observed CTR comparisons

A higher observed CTR is not automatically a statistically established winner. These comparisons use Holm-adjusted p-values across all campaign pairs.


In [ ]:
from IPython.display import display

pairs_df = pairwise_comparisons(df)
display(pairs_df)


## Daily CTR by campaign

The date split below is exploratory; the notebook does not establish that a platform algorithm changed.


In [ ]:
# Prepare daily time series data (if not already defined)
daily_ts = (
    df
    .groupby(['campaign_name','date'])
    .agg(clicks=('clicks_to_landing_page','sum'),
         impressions=('impressions','sum'))
    .reset_index()
)
daily_ts = daily_ts.loc[daily_ts['impressions'] > 0].copy()
daily_ts['ctr'] = daily_ts['clicks'] / daily_ts['impressions']

# Define distinct styles for each campaign
styles = {
    'AI':             {'color': '#1f77b4', 'linestyle': '-',  'marker': 'o'},
    'Analytics':      {'color': '#2ca02c', 'linestyle': '-', 'marker': 's'},
    'Digitize':       {'color': '#d62728', 'linestyle': '-',  'marker': '^'},
    'Loss Prevention':{'color': '#9467bd', 'linestyle': '-', 'marker': 'd'},
}

plt.figure(figsize=(12, 6))
for camp, style in styles.items():
    sub = daily_ts[daily_ts['campaign_name'] == camp]
    plt.plot(
        sub['date'], sub['ctr'],
        label=camp,
        color=style['color'],
        linestyle=style['linestyle'],
        marker=style['marker'],
        markersize=6,
        linewidth=1
    )

plt.title('Daily CTR Time Series by Campaign', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Click-Through Rate', fontsize=12)
plt.grid(True, linestyle='-', alpha=0.3)
plt.legend(title='Campaign', bbox_to_anchor=(0.3, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Pre vs Post 7/25 CTR Visual Comparison

In [ ]:
split_date = pd.Timestamp('2025-07-25')
periods = [
    ('Pre-2025-07-25', df[df['date'] < split_date]),
    ('Post-2025-07-25', df[df['date'] >= split_date])
]
for label, subset in periods:
    summary = campaign_summary(subset)
    print(label)
    if summary.empty:
        print('No positive-impression observations in this period.')
        continue
    display(summary.reset_index())
    plt.figure(figsize=(8, 5))
    plt.barh(summary.index, summary['ctr'], xerr=[
        (summary['ctr'] - summary['ci_lower']).clip(lower=0),
        (summary['ci_upper'] - summary['ctr']).clip(lower=0)
    ])
    plt.xlabel('Click-through rate')
    plt.title(f'Campaign CTRs with 95% Wilson intervals: {label}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()


## 7. Overall ranking and pairwise comparisons


In [ ]:
agg = campaign_summary(df)
pairs_df = pairwise_comparisons(df)
display(agg.reset_index())
display(pairs_df)


## Pre vs Post 7/25 Ranking

In [ ]:
results = {}
for label, subset in periods:
    period_summary = campaign_summary(subset)
    period_pairs = pairwise_comparisons(subset)
    print(label)
    display(period_summary.reset_index())
    display(period_pairs)
    results[label] = (period_summary, period_pairs)


## 8. CTR Comparison via ANOVA

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt

# 8.1 Compute daily CTR proportions
daily = (
    df
    .groupby(['campaign_name','date'])
    .agg(clicks=('clicks_to_landing_page','sum'),
         impressions=('impressions','sum'))
    .reset_index()
)
daily = daily.loc[daily['impressions'] > 0].copy()
daily['ctr'] = daily['clicks'] / daily['impressions']

# 8.2 One-way ANOVA across campaigns
group_ctrs = [grp['ctr'].values for _, grp in daily.groupby('campaign_name')]
if len(group_ctrs) >= 2 and all(len(group) >= 2 for group in group_ctrs):
    f_stat, p_val = stats.f_oneway(*group_ctrs)
    print(f'Exploratory ANOVA: F = {f_stat:.3f}, p = {p_val:.3f}')
else:
    print('ANOVA skipped: at least two campaigns with multiple observations are needed.')

# 8.4 (Optional) Boxplot of daily CTR by campaign
plt.figure(figsize=(8,5))
daily.boxplot(column='ctr', by='campaign_name', grid=False)
plt.suptitle('')
plt.title('Daily CTR Distribution by Campaign')
plt.ylabel('CTR')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 9. Box-plot for Pre- vs Post- July 25

In [ ]:
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Prepare daily CTR data
daily = (
    df
    .groupby(['campaign_name', 'date'])
    .agg(clicks=('clicks_to_landing_page', 'sum'),
         impressions=('impressions', 'sum'))
    .reset_index()
)
daily = daily.loc[daily['impressions'] > 0].copy()
daily['ctr'] = daily['clicks'] / daily['impressions']

# Ensure date is datetime
daily['date'] = pd.to_datetime(daily['date'])

# Split into pre-7/25 and post-7/25
split_date = pd.to_datetime('2025-07-25')
daily_pre = daily[daily['date'] < split_date]
daily_post = daily[daily['date'] >= split_date]

def plot_period_distribution(df_subset, label):
    print(f"\n--- {label} (n_days={df_subset['date'].nunique()}) ---")
    
    if df_subset.empty:
        print('No positive-impression observations in this period.')
        return

    # Boxplot
    plt.figure(figsize=(8, 4))
    df_subset.boxplot(column='ctr', by='campaign_name', grid=False)
    plt.suptitle('')
    plt.title(f'Daily CTR by Campaign ({label})')
    plt.ylabel('CTR')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Run for pre-7/25
plot_period_distribution(daily_pre, 'Pre-2025-07-25')

# Run for post-7/25
plot_period_distribution(daily_post, 'Post-2025-07-25')
